# 04 · MCP tools

Exposes tau's retrieval primitives as MCP tools (`src/tau/mcp/server.py`) and points the LangGraph agent's retrieval step at them instead of calling `tau.retrieval.search` directly (`build_temporal_agent_graph_mcp` in [src/tau/agent/graph.py](../src/tau/agent/graph.py)).

MCP is purely an interface layer here — no retrieval logic is duplicated. Reranking (`rerank_results`) and tau decay (`apply_temporal_decay`) are not MCP tools (out of scope) and still run as direct Python calls inside the graph.

Assumes notebook 02 has already been run at least once, so Postgres' `documents` table is populated with embeddings.

**Kernel must be `Python (tau)`** (this project's `.venv`). This notebook uses `await` at the top level of cells (MCP's client API is async) — that's natively supported by Jupyter/IPython, no extra setup needed.

## 1. Setup

In [1]:
import sys
from pathlib import Path

print("PYTHON:", sys.executable)

import tau
print("tau package:", Path(tau.__file__).parent)


PYTHON: /Users/akshay/tau/.venv/bin/python
tau package: /Users/akshay/tau/src/tau


In [2]:
import os
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
PG_DSN = os.environ.get("PG_DSN", "dbname=tau")

print("VOYAGE_API_KEY loaded:", bool(VOYAGE_API_KEY))
print("PG_DSN:", PG_DSN)


VOYAGE_API_KEY loaded: True
PG_DSN: dbname=tau


In [3]:
import psycopg
from pgvector.psycopg import register_vector

pg_conn = psycopg.connect(PG_DSN, autocommit=True)
register_vector(pg_conn)

row_count = pg_conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
print("Connected to Postgres:", pg_conn.info.dbname)
print("documents row count:", row_count)

assert row_count > 0, (
    "documents table is empty — run notebook 02 first to embed and load documents"
)


Connected to Postgres: tau
documents row count: 114


## 2. What MCP is doing in this project

`build_mcp_server(pg_conn, voyage_api_key)` (in [src/tau/mcp/server.py](../src/tau/mcp/server.py)) builds an `MCPServer` (the `mcp` Python SDK's tool server) exposing exactly three typed tools — each one a thin wrapper around a function that already existed:

| tool | schema | wraps |
|---|---|---|
| `search_semantic` | `query: str, k: int = 20` | `embed_query` + `semantic_search` (no time filter) |
| `search_recent` | `query: str, start_time: str, end_time: str, k: int = 20` | `embed_query` + `semantic_search(..., time_window=...)` |
| `get_document` | `document_id: str` | `tau.retrieval.search.get_document` |

`start_time`/`end_time` are ISO-8601 strings (MCP tool arguments are JSON, which has no native datetime type) and `published_at` comes back the same way — the graph converts it back to a `datetime` after the call.

**Transport:** the simplest local setup — `mcp.Client(mcp_server)` connects directly to an in-process `MCPServer` instance over an in-memory transport. Same protocol (tool discovery, JSON-schema-validated calls, structured results) as talking to a server over stdio/subprocess, just without spawning one — no separate process, no Docker, nothing to host.

**What changed in the agent:** `build_temporal_agent_graph_mcp` (new, alongside the original `build_temporal_agent_graph`) replaces the `retrieve` node's direct `semantic_search`/`embed_query` calls with an `await mcp_client.call_tool(...)` — `search_recent` when `classify_intent` extracted a `time_window` (`explicit_temporal`), `search_semantic` otherwise (`current` / `topical`). Routing, reranking, and tau decay are untouched.

## 3. Start/connect to MCP server

In [4]:
from mcp import Client

from tau.mcp import build_mcp_server

mcp_server = build_mcp_server(pg_conn, VOYAGE_API_KEY)

mcp_client = Client(mcp_server)
await mcp_client.__aenter__()  # kept open across the cells below; closed in the summary section

print("MCP client connected to server:", mcp_server.name)


MCP client connected to server: tau-retrieval


## 4. List/discover available tools

In [5]:
tools = await mcp_client.list_tools()

for t in tools.tools:
    print(f'- {t.name}: {t.description}')
    print(f'  schema: {t.input_schema}')


- search_semantic: Semantic top-k search over the whole corpus (pgvector cosine similarity).

Use for queries with no explicit time constraint.

  schema: {'type': 'object', 'properties': {'query': {'title': 'Query', 'type': 'string'}, 'k': {'default': 20, 'title': 'K', 'type': 'integer'}}, 'required': ['query'], 'title': 'search_semanticArguments'}
- search_recent: Semantic top-k search hard-filtered to published_at in [start_time, end_time].

start_time / end_time are ISO-8601 datetimes. Use for explicit temporal queries
(e.g. "in the last 3 hours") where the extracted window is a hard constraint,
not just a ranking signal.

  schema: {'type': 'object', 'properties': {'query': {'title': 'Query', 'type': 'string'}, 'start_time': {'title': 'Start Time', 'type': 'string'}, 'end_time': {'title': 'End Time', 'type': 'string'}, 'k': {'default': 20, 'title': 'K', 'type': 'integer'}}, 'required': ['query', 'start_time', 'end_time'], 'title': 'search_recentArguments'}
- get_document: Look up 

## 5. Call `search_semantic` directly

In [6]:
result = await mcp_client.call_tool("search_semantic", {"query": "OpenAI safety", "k": 5})
documents = result.structured_content["result"]

print("is_error:", result.is_error)
print("documents returned:", len(documents))
for doc in documents:
    print(f'  - {doc["title"]!r}  source={doc["source"]}  published_at={doc["published_at"]}')


is_error: False
documents returned: 5
  - 'The Safety Reckoning Inside OpenAI'  source=wired  published_at=2026-08-13T18:37:19-04:00
  - 'OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'  source=wired  published_at=2026-08-18T14:33:11-04:00
  - 'OpenAI lays out new security changes after its AI hacked Hugging Face'  source=hackernews  published_at=2026-08-18T20:07:06-04:00
  - "OpenAI's overhead will rise 20 percent for some workloads as it hardens security"  source=hackernews  published_at=2026-08-18T20:14:58-04:00
  - 'Rogue AI Agents Aren’t Evil. They’re Just Eager to Please'  source=wired  published_at=2026-08-12T14:45:00-04:00


## 6. Call `search_recent` directly

In [7]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)
start_time = now - timedelta(hours=3)

result = await mcp_client.call_tool(
    "search_recent",
    {
        "query": "OpenAI",
        "start_time": start_time.isoformat(),
        "end_time": now.isoformat(),
        "k": 5,
    },
)


In [8]:
documents = result.structured_content["result"]

print("is_error:", result.is_error)
print(f'window: [{start_time}, {now}]')
print("documents returned:", len(documents))
for doc in documents:
    print(f'  - {doc["title"]!r}  source={doc["source"]}  published_at={doc["published_at"]}')


is_error: False
window: [2026-08-18 22:58:07.621861+00:00, 2026-08-19 01:58:07.621861+00:00]
documents returned: 5
  - 'OpenAI lays out new security changes after its AI hacked Hugging Face'  source=hackernews  published_at=2026-08-18T20:07:06-04:00
  - "OpenAI's overhead will rise 20 percent for some workloads as it hardens security"  source=hackernews  published_at=2026-08-18T20:14:58-04:00
  - 'Show HN: Interactive, animated architecture of any HuggingFace models'  source=hackernews  published_at=2026-08-18T19:57:36-04:00
  - "How to tame AI's voracious appetite for energy"  source=hackernews  published_at=2026-08-18T20:14:34-04:00
  - 'Artificial intelligence boosts automated biolabs'  source=hackernews  published_at=2026-08-18T19:59:29-04:00


## 7. Call `get_document` directly

In [9]:
sample_id = documents[0]["id"] if documents else None
print("looking up id:", sample_id)

result = await mcp_client.call_tool("get_document", {"document_id": sample_id})
doc = result.structured_content["result"]
print("found:", doc["title"] if doc else None)

missing = await mcp_client.call_tool("get_document", {"document_id": "does-not-exist"})
print("missing id result:", missing.structured_content["result"])


looking up id: b3348607767e70eea99ff218a3d4077e19c92b5ef4aabb5c63ef45fa4ce4ee89
found: OpenAI lays out new security changes after its AI hacked Hugging Face
missing id result: None


## 8. Run LangGraph agent on three query types

Same three routes as notebook 03, now retrieved through the MCP tools instead of direct `tau.retrieval.search` calls.

In [10]:
from tau.agent.graph import build_temporal_agent_graph_mcp

agent_graph = build_temporal_agent_graph_mcp(
    mcp_client,
    VOYAGE_API_KEY,
    limit=20,
    top_k=5,
    tau_hours=24,
)

test_queries = [
    "What happened with OpenAI in the last 3 hours?",  # explicit_temporal
    "What's going on with OpenAI?",                     # current
    "Explain OpenAI's safety approach",                 # topical
]

routing_states = {}

for query in test_queries:
    state = await agent_graph.ainvoke({"query": query})
    routing_states[state["route"]] = state

    print(f'query={state["query"]!r}')
    print(f'  route={state["route"]}  time_window={state["time_window"]}')
    print(f'  mcp_tool={state["mcp_tool"]}  mcp_arguments={state["mcp_arguments"]}')
    print()

    for r in state["final_results"]:
        print(f'    - title={r["title"]!r}')
        print(f'      source={r["source"]}  published_at={r["published_at"]}')

        score_line = f'      rerank_score={r["rerank_score"]:.4f}'
        if "final_score" in r:
            score_line += (
                f'  recency_weight={r["recency_weight"]:.4f}'
                f'  final_score={r["final_score"]:.4f}'
            )
        print(score_line)

    print()


query='What happened with OpenAI in the last 3 hours?'
  route=explicit_temporal  time_window=(datetime.datetime(2026, 8, 18, 22, 58, 9, 44632, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 8, 19, 1, 58, 9, 44632, tzinfo=datetime.timezone.utc))
  mcp_tool=search_recent  mcp_arguments={'query': 'What happened with OpenAI in the last 3 hours?', 'start_time': '2026-08-18T22:58:09.044632+00:00', 'end_time': '2026-08-19T01:58:09.044632+00:00', 'k': 20}

    - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
      source=hackernews  published_at=2026-08-18 20:07:06-04:00
      rerank_score=0.6797
    - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
      source=hackernews  published_at=2026-08-18 20:14:58-04:00
      rerank_score=0.4805
    - title='Could End the RAM Crisis [video]'
      source=hackernews  published_at=2026-08-18 20:15:19-04:00
      rerank_score=0.3320
    - title='Cerebras CS-4 rack systems ju

query="What's going on with OpenAI?"
  route=current  time_window=None
  mcp_tool=search_semantic  mcp_arguments={'query': "What's going on with OpenAI?", 'k': 20}

    - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
      source=hackernews  published_at=2026-08-18 20:07:06-04:00
      rerank_score=0.7422  recency_weight=0.9251  final_score=0.6866
    - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
      source=hackernews  published_at=2026-08-18 20:14:58-04:00
      rerank_score=0.5977  recency_weight=0.9302  final_score=0.5559
    - title='OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'
      source=wired  published_at=2026-08-18 14:33:11-04:00
      rerank_score=0.7266  recency_weight=0.7337  final_score=0.5330
    - title='The Safety Reckoning Inside OpenAI'
      source=wired  published_at=2026-08-13 18:37:19-04:00
      rerank_score=0.6719  recency_weight=0.0059  final_score=0.0039
    - 

query="Explain OpenAI's safety approach"
  route=topical  time_window=None
  mcp_tool=search_semantic  mcp_arguments={'query': "Explain OpenAI's safety approach", 'k': 20}

    - title='OpenAI Overhauls Safety Protocols After Its AI Agents Went Rogue'
      source=wired  published_at=2026-08-18 14:33:11-04:00
      rerank_score=0.6875
    - title='The Safety Reckoning Inside OpenAI'
      source=wired  published_at=2026-08-13 18:37:19-04:00
      rerank_score=0.6250
    - title='OpenAI lays out new security changes after its AI hacked Hugging Face'
      source=hackernews  published_at=2026-08-18 20:07:06-04:00
      rerank_score=0.4980
    - title="OpenAI's overhead will rise 20 percent for some workloads as it hardens security"
      source=hackernews  published_at=2026-08-18 20:14:58-04:00
      rerank_score=0.4590
    - title='The White House Is Going to Expand Its AI Policy'
      source=wired  published_at=2026-08-12 17:00:00-04:00
      rerank_score=0.3711



## 9. Show which MCP tool the agent selected

In [11]:
for route_name, state in routing_states.items():
    print(f'{route_name}: tool={state["mcp_tool"]}  arguments={state["mcp_arguments"]}')


explicit_temporal: tool=search_recent  arguments={'query': 'What happened with OpenAI in the last 3 hours?', 'start_time': '2026-08-18T22:58:09.044632+00:00', 'end_time': '2026-08-19T01:58:09.044632+00:00', 'k': 20}
current: tool=search_semantic  arguments={'query': "What's going on with OpenAI?", 'k': 20}
topical: tool=search_semantic  arguments={'query': "Explain OpenAI's safety approach", 'k': 20}


In [12]:
# --- explicit_temporal must use search_recent -------------------------------
explicit_state = routing_states["explicit_temporal"]
assert explicit_state["mcp_tool"] == "search_recent", (
    f'explicit_temporal used {explicit_state["mcp_tool"]!r}, expected search_recent'
)
print("PASS: explicit_temporal -> search_recent")

# --- current must use search_semantic ----------------------------------------
current_state = routing_states["current"]
assert current_state["mcp_tool"] == "search_semantic", (
    f'current used {current_state["mcp_tool"]!r}, expected search_semantic'
)
print("PASS: current -> search_semantic")

# --- topical must use search_semantic ----------------------------------------
topical_state = routing_states["topical"]
assert topical_state["mcp_tool"] == "search_semantic", (
    f'topical used {topical_state["mcp_tool"]!r}, expected search_semantic'
)
print("PASS: topical -> search_semantic")

# --- explicit_temporal: no tau decay -----------------------------------------
for r in explicit_state["final_results"]:
    assert "recency_weight" not in r and "final_score" not in r, (
        f'{r["title"]!r} unexpectedly carries tau-decay fields for explicit_temporal'
    )
print("PASS: explicit_temporal — no tau decay applied")

# --- current: tau decay applied -----------------------------------------------
for r in current_state["final_results"]:
    assert "recency_weight" in r and "final_score" in r, (
        f'{r["title"]!r} is missing tau-decay fields for the current route'
    )
print("PASS: current — tau decay applied")

# --- topical: no tau decay ------------------------------------------------------
for r in topical_state["final_results"]:
    assert "recency_weight" not in r and "final_score" not in r, (
        f'{r["title"]!r} unexpectedly carries tau-decay fields for topical'
    )
print("PASS: topical — no tau decay applied")

print()
print("All MCP routing assertions passed.")


PASS: explicit_temporal -> search_recent
PASS: current -> search_semantic
PASS: topical -> search_semantic
PASS: explicit_temporal — no tau decay applied
PASS: current — tau decay applied
PASS: topical — no tau decay applied

All MCP routing assertions passed.


## 10. Summary

MCP now sits between the LangGraph agent and Postgres/Voyage as a thin, typed tool interface — nothing about retrieval, reranking, or tau decay changed underneath it:

| route | MCP tool used | tau decay |
|---|---|---|
| `explicit_temporal` | `search_recent` | no |
| `current` | `search_semantic` | yes (`tau_hours=24`) |
| `topical` | `search_semantic` | no |

All three MCP tools (`search_semantic`, `search_recent`, `get_document`) were exercised directly (sections 5–7) and through the agent (section 8), and all routing/tau assertions passed. `tau.retrieval.*` and `tau.agent.graph` remain unmodified in their non-MCP form — `build_temporal_agent_graph_mcp` sits alongside the original `build_temporal_agent_graph`, so notebook 03 still works exactly as before.

**Stopping here per scope** — no answer generation, no evals, no multi-agent setup, no memory. Those are explicitly out of scope for this notebook.

In [13]:
try:
    await mcp_client.__aexit__(None, None, None)
except Exception as e:
    # Jupyter runs each cell's top-level `await` in its own asyncio Task, but the
    # in-process MCP transport's background reader lives in an anyio task group
    # tied to the Task that entered it (section 3's cell). Exiting from a
    # different cell/Task trips anyio's cancel-scope check. Harmless here — the
    # in-process transport has no subprocess/socket to leak; a real stdio/SSE
    # server would instead be closed from the same task/cell that started it.
    print(f"Note: MCP client cleanup hit a Jupyter cross-cell task quirk ({e!r}); "
          "harmless for the in-process transport, nothing left to leak.")

pg_conn.close()
print("Postgres connection closed.")


Note: MCP client cleanup hit a Jupyter cross-cell task quirk (RuntimeError('Attempted to exit cancel scope in a different task than it was entered in')); harmless for the in-process transport, nothing left to leak.
Postgres connection closed.
